[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Merging


## What you will be able to do

Join two tables on a key column, choose which unmatched rows the result keeps, and catch a
duplicated key before it multiplies your rows.

## The idea

### The problem

The readings say which station each reading came from, and nothing else about the station. Its
latitude is in `stations`, a second table with one row for each station. A question such as "how
cold were the stations north of 65 degrees?" needs the readings for the values and the latitudes to
decide which stations count.

Copying the latitudes across by hand means a dictionary from station to latitude and a loop over
35,040 readings, and each mismatch has to be decided as it comes. Kirkenes has a latitude and no
readings. A station whose name was typed differently in the two tables matches nothing. A station
that appears twice in `stations`, say because a corrected latitude was added without removing the
old one, gives every one of its readings two latitudes to choose from.

### What a merge is

> A **merge**, or **join**, combines two tables by matching the values in one or more **key**
> columns. Each row of the result pairs a row of the left table with a row of the right table whose
> keys are equal. The argument `how` decides what happens to rows that match nothing: `"inner"`
> drops them, `"left"` keeps every row of the left table, `"right"` every row of the right table,
> and `"outer"` every row of both, with `NaN` in the columns the other table would have filled.

### Why it works that way

A merge makes one output row for every pair of matching rows. When each key appears once on the
right, as each station does in `stations`, every reading finds one latitude, and the result has as
many rows as the readings. This is a **many-to-one** merge, many readings to one station. When a key
appears twice on the right, every row on the left that has that key matches both, and the result
gains a row for each extra match, with no warning. `validate="many_to_one"` checks the right table's
keys before merging and raises when one repeats.

The key columns are compared by value, so they must hold the same kind of value. A station number
stored as the integer 4 in one table and the text `"4"` in another never matches, and pandas raises
rather than return a merge in which nothing matched.

`pd.concat` is the other way to combine tables. It stacks them, one after another, and lines up
their columns by name. A merge adds columns to rows that share a key, and `concat` adds rows to
columns that share a name.

### Where this shows up

SQL calls it a join, and the **Joins** notebook in the **sqlite3, Deep Dive** guide covers the same
four kinds in a database. Looking up a value in a second sheet of a spreadsheet is a left join done
one cell at a time. Any analysis that combines a table of measurements with a table describing what
was measured, such as orders and customers or readings and stations, starts with a merge. The **Why
Pandas** notebook combined two Series in another way, by lining up their indexes.

### What this notebook covers

- joining the stations' latitudes to each station's mean
- the four kinds of join, and `indicator=True` to see where each row came from
- a many-to-one merge of every reading with its station, checked with `validate`
- `suffixes` for columns that share a name
- `concat` for stacking tables with the same columns
- four errors: a duplicated key that multiplies rows, `validate` catching it, a key of the wrong
  type, and `concat` with columns that do not line up

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

stations = pd.DataFrame({"station": ["Bergen", "Oslo", "Kirkenes"], "latitude": [60.39, 59.91, 69.73]})
means = pd.DataFrame({"station": ["Bergen", "Oslo"], "mean": [8.0, 6.5]})
print(pd.merge(stations, means, on="station"))
print(pd.merge(stations, means, on="station", how="left"))
```

```
  station  latitude  mean
0  Bergen     60.39   8.0
1    Oslo     59.91   6.5
    station  latitude  mean
0    Bergen     60.39   8.0
1      Oslo     59.91   6.5
2  Kirkenes     69.73   NaN
```

`on="station"` names the key. The first merge kept only the stations found in both tables, so
Kirkenes, which has no mean, is gone. `how="left"` kept every row of the left table, and Kirkenes is
back with `NaN` for the mean it does not have.

## Setup

One library, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide, with Svalbard's 24 readings for 2 March missing.
`readings` holds all 35,040, and `stations` holds each station's latitude from `LATITUDES`, with
Kirkenes, a station that has no readings.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### The stations and their means

`stations` has five rows, and `means`, which holds a mean for each station that sent readings, has
four:

In [2]:
means = readings.groupby("station")["celsius"].mean().round(2).reset_index(name="mean")

print(stations)
print()
print(means)

    station  latitude
0    Bergen     60.39
1      Oslo     59.91
2  Svalbard     78.22
3    Tromso     69.65
4  Kirkenes     69.73

    station  mean
0    Bergen  8.00
1      Oslo  6.50
2  Svalbard -4.48
3    Tromso  3.50


`reset_index(name="mean")` turned the groupby's Series into a table with a `station` column, so both
tables now have a column called `station` to join on.

### An inner join and a left join

`pd.merge` joins on the column named in `on`, and an inner join is what it does unless told
otherwise:

In [3]:
pd.merge(stations, means, on="station")

,station,latitude,mean
0,Bergen,60.39,8.00
1,Oslo,59.91,6.50
2,Svalbard,78.22,-4.48
3,Tromso,69.65,3.50


The result has four rows. Kirkenes is in `stations` and not in `means`, so the inner join dropped
it. A left join keeps every row of the left table, `stations`:

In [4]:
pd.merge(stations, means, on="station", how="left")

,station,latitude,mean
0,Bergen,60.39,8.00
1,Oslo,59.91,6.50
2,Svalbard,78.22,-4.48
3,Tromso,69.65,3.50
4,Kirkenes,69.73,NaN


### Inner, left, right and outer joins

Every station in `means` is in `stations`, so the four kinds of join can give only two answers. A
table of the stations on the mainland leaves out Svalbard, an archipelago, which makes each kind of
join keep a different set of rows:

In [5]:
mainland = stations[stations["station"] != "Svalbard"]

for how in ["inner", "left", "right", "outer"]:
    result = pd.merge(mainland, means, on="station", how=how)
    print(f"{how:>5}: {len(result)} rows", result["station"].tolist())

inner: 3 rows ['Bergen', 'Oslo', 'Tromso']
 left: 4 rows ['Bergen', 'Oslo', 'Tromso', 'Kirkenes']
right: 4 rows ['Bergen', 'Oslo', 'Svalbard', 'Tromso']
outer: 5 rows ['Bergen', 'Kirkenes', 'Oslo', 'Svalbard', 'Tromso']


Kirkenes is only in `mainland`, the left table, and Svalbard is only in `means`, the right table.
The inner join keeps the three stations in both. The left join adds Kirkenes, the right join adds
Svalbard, and the outer join keeps all five, sorted by station name. `indicator=True` adds a column
saying where each row of the result came from:

In [6]:
pd.merge(mainland, means, on="station", how="outer", indicator=True)

,station,latitude,mean,_merge
0,Bergen,60.39,8.00,both
1,Kirkenes,69.73,NaN,left_only
2,Oslo,59.91,6.50,both
3,Svalbard,NaN,-4.48,right_only
4,Tromso,69.65,3.50,both


`left_only` and `right_only` mark the two rows that matched nothing. Kirkenes has no mean, and
Svalbard has no latitude in `mainland`, so `NaN` fills the columns the other table would have
filled. Filtering on `_merge` is a quick way to list what failed to match.

### Every reading with its latitude

Merging the readings with `stations` gives each of the 35,040 readings the latitude of its station.
Each station appears once in `stations`, so this is a many-to-one merge, and `validate` checks that
before merging:

In [7]:
located = pd.merge(readings, stations, on="station", how="left", validate="many_to_one")

print(located.shape)
located.head(3)

(35040, 4)


,station,time,celsius,latitude
0,Bergen,2025-01-01,-3.6,60.39
1,Oslo,2025-01-01,-3.5,59.91
2,Svalbard,2025-01-01,-14.6,78.22


The result has the readings' 35,040 rows and one more column. Kirkenes matched no reading, and a
left join on the readings drops it, since it is not in the left table. With the latitudes in place,
the question from the start of the notebook is a filter and a mean:

In [8]:
north = located[located["latitude"] > 65]

print(north["station"].unique().tolist())
print("mean north of 65 degrees:", round(north["celsius"].mean(), 2))

['Svalbard', 'Tromso']
mean north of 65 degrees: -0.49


### Columns that share a name

January's and July's means come out of the same code, so both tables call their column `celsius`.
A merge cannot put two columns of the same name side by side, so it adds `_x` and `_y`:

In [9]:
month = readings["time"].dt.month
january = readings[month == 1].groupby("station")["celsius"].mean().round(1).reset_index()
july = readings[month == 7].groupby("station")["celsius"].mean().round(1).reset_index()

pd.merge(january, july, on="station")

,station,celsius_x,celsius_y
0,Bergen,-0.9,16.9
1,Oslo,-2.4,15.4
2,Svalbard,-13.4,4.4
3,Tromso,-5.4,12.4


`suffixes` chooses the endings, which makes the result readable without looking back at the code:

In [10]:
seasons = pd.merge(january, july, on="station", suffixes=("_january", "_july"))
seasons.assign(swing=(seasons["celsius_july"] - seasons["celsius_january"]).round(1))

,station,celsius_january,celsius_july,swing
0,Bergen,-0.9,16.9,17.8
1,Oslo,-2.4,15.4,17.8
2,Svalbard,-13.4,4.4,17.8
3,Tromso,-5.4,12.4,17.8


Every station's July is 17.8 degrees warmer than its January, because the formula gives each station
the same seasonal swing around a different mean.

### Stacking tables with concat

`pd.concat` puts tables one after another. With a column saying which month each row is for, the two
tables become one long table, and `ignore_index=True` numbers its rows again from 0:

In [11]:
pd.concat([january.assign(month="January"), july.assign(month="July")], ignore_index=True)

,station,celsius,month
0,Bergen,-0.9,January
1,Oslo,-2.4,January
2,Svalbard,-13.4,January
3,Tromso,-5.4,January
4,Bergen,16.9,July
5,Oslo,15.4,July
6,Svalbard,4.4,July
7,Tromso,12.4,July


`concat` matched the columns by name, and both tables have `station` and `celsius`, so every row is
complete. The Common errors section shows what happens when the names differ.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/15-merging-solutions.ipynb).

**1.** Merge `means` and `stations` so that every station appears, Kirkenes included, with `means` as the left table.

In [12]:
# your code here

**2.** Using `located`, print the mean reading of the stations south of 61 degrees.

In [13]:
# your code here

**3.** Make an outer merge of `mainland` and `means` with `indicator=True`, and show only the rows that came from one table alone.

In [14]:
# your code here

**4.** Make a table of each station's coldest and warmest reading of the year, from two groupby results merged on `station` with suffixes `_coldest` and `_warmest`.

In [15]:
# your code here

**5.** Stack January's and July's means with `concat`, with a `month` column, and print the station and month of the lowest mean.

In [16]:
# your code here

## Common errors

### No error, and Tromso's readings counted twice

A corrected latitude for Tromso was added to the end of the table, and the old row was left in:

In [17]:
corrected = pd.concat([stations, pd.DataFrame({"station": ["Tromso"], "latitude": [69.68]})], ignore_index=True)
doubled = pd.merge(readings, corrected, on="station")

print(len(readings), "readings,", len(doubled), "rows after the merge")
print("mean of every reading:", round(readings["celsius"].mean(), 2),
      "| after the merge:", round(doubled["celsius"].mean(), 2))
doubled["station"].value_counts().sort_index()

35040 readings, 43800 rows after the merge
mean of every reading: 3.38 | after the merge: 3.41


station
Bergen       8760
Oslo         8760
Svalbard     8760
Tromso      17520
Name: count, dtype: int64

Tromso appears twice in `corrected`, so each of its 8,760 readings matched both rows, and the merge
made 17,520 rows for it. Nothing failed. A count of Tromso's readings is now twice the truth, and
the mean of every reading moved from 3.38 to 3.41, because Tromso now has twice the weight of each
other station. `duplicated("station", keep=False)` marks every row whose station appears more than
once, and `drop_duplicates("station", keep="last")` keeps the correction, after which the merge has
one row for each reading again:

In [18]:
print(corrected[corrected.duplicated("station", keep=False)])

fixed = corrected.drop_duplicates("station", keep="last")
print(len(pd.merge(readings, fixed, on="station", validate="many_to_one")), "rows after the merge")

  station  latitude
3  Tromso     69.65
5  Tromso     69.68
35040 rows after the merge


### MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

In [19]:
pd.merge(readings, corrected, on="station", validate="many_to_one")

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

`validate` turns the silent error above into a loud one. It checked the right table before merging,
found Tromso twice, and raised. Pass it on every merge whose shape you know: `"one_to_one"`,
`"one_to_many"` or `"many_to_one"`.

### ValueError: You are trying to merge on int64 and object columns for key 'station_id'. If you wish to proceed you should use pd.concat


Here the stations have numbers, and a table of regions keyed by the same numbers came from a file
read with `dtype=str`, so its numbers are text:

In [20]:
numbered = stations.assign(station_id=range(1, 6))
regions = pd.DataFrame({"station_id": ["1", "2", "3", "4", "5"],
                        "region": ["Vestland", "Oslo", "Svalbard", "Troms", "Finnmark"]})

pd.merge(numbered, regions, on="station_id")

ValueError: You are trying to merge on int64 and object columns for key 'station_id'. If you wish to proceed you should use pd.concat

The integer 1 and the text `"1"` are different values, so no key on one side equals a key on the
other, and pandas raises instead. The message's suggestion of `pd.concat` does not apply here,
because the two tables are meant to be joined. Convert one side so the types agree, with
`regions.astype({"station_id": int})`, and the merge works:

In [21]:
pd.merge(numbered, regions.astype({"station_id": int}), on="station_id")

,station,latitude,station_id,region
0,Bergen,60.39,1,Vestland
1,Oslo,59.91,2,Oslo
2,Svalbard,78.22,3,Svalbard
3,Tromso,69.65,4,Troms
4,Kirkenes,69.73,5,Finnmark


### No error, and a NaN in every row

July's means came from code that called the column `mean` instead of `celsius`:

In [22]:
july_by_other_code = july.rename(columns={"celsius": "mean"})

pd.concat([january, july_by_other_code])

,station,celsius,mean
0,Bergen,-0.9,NaN
1,Oslo,-2.4,NaN
2,Svalbard,-13.4,NaN
3,Tromso,-5.4,NaN
0,Bergen,NaN,16.9
1,Oslo,NaN,15.4
2,Svalbard,NaN,4.4
3,Tromso,NaN,12.4


`concat` lines up columns by name. `celsius` exists only in the first table and `mean` only in the
second, so each became a column of its own, with `NaN` wherever a table had no such column. The row
labels repeat, too, since each table kept its own 0 to 3. Rename the columns so they agree, and pass
`ignore_index=True`.

## Recap

- Join two tables on a key column with `pd.merge(..., on=...)`
- Choose with `how` which unmatched rows to keep: `"inner"`, `"left"`, `"right"` or `"outer"`
- See where each row came from with `indicator=True`
- Check a merge's shape with `validate`, which catches a duplicated key before it multiplies rows
- Name clashing columns with `suffixes`, and stack tables that share their columns with `concat`

## What is next

The **Dates and Times** notebook parses times from text, shows the readings in Oslo's time zone,
where the clocks change twice a year, and averages them by day with `resample`.

---

&#8592; **Previous:** [Reshaping](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/14-reshaping.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Dates and Times](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/16-dates-and-times.ipynb) &#8594;
